In [ ]:
!pip install datasets librosa soundfile pandas tqdm jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.8 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

# Dataset Download

In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Audio
from tqdm import tqdm
import soundfile as sf
import librosa
from huggingface_hub import login

# ---------------- CONFIG ----------------
DATASET_NAME = "ai4bharat/IndicVoices"
LANG = "hindi"
OUT_DIR = "benchmark_data"
TARGET_SAMPLE_RATE = 16000

# Updated Buckets to be slightly more flexible
BUCKETS = {
    "5s": (4, 8),
    "15s": (12, 20),
    "30s": (20, 35)
}
CLIPS_PER_BUCKET = 10

# ----------------------------------------

def prepare_folders():
    for name in BUCKETS:
        os.makedirs(os.path.join(OUT_DIR, name), exist_ok=True)

def get_audio_data(token):
    print(f"Streaming {DATASET_NAME} ({LANG})...")
    login(token)

    # 1. Load and 2. Cast the column immediately
    ds = load_dataset(DATASET_NAME, LANG, split="train", streaming=True)
    ds = ds.cast_column("audio_filepath", Audio(sampling_rate=TARGET_SAMPLE_RATE))

    counts = {k: 0 for k in BUCKETS}
    metadata = []

    # Using enumerate to track progress in the terminal
    for i, example in enumerate(tqdm(ds, desc="Sourcing Audio")):

        # 3. Filter Case-Insensitive
        scenario = example.get("scenario", "").lower()
        if scenario not in ["extempore", "conversation", "conversational"]:
            continue

        # 4. Access the Audio Dictionary
        audio_data = example.get("audio_filepath")
        if not audio_data:
            continue

        y = audio_data["array"]
        sr = audio_data["sampling_rate"]
        duration = len(y) / sr

        # 5. Assign to Bucket
        assigned_bucket = None
        for name, (min_d, max_d) in BUCKETS.items():
            if min_d <= duration <= max_d and counts[name] < CLIPS_PER_BUCKET:
                assigned_bucket = name
                break

        if not assigned_bucket:
            continue

        # 6. Save the File
        file_name = f"clip_{counts[assigned_bucket]}.wav"
        file_path = os.path.join(OUT_DIR, assigned_bucket, file_name)

        sf.write(file_path, y, TARGET_SAMPLE_RATE)

        # 7. Use the CLEAN 'normalized' text for the Ground Truth
        metadata.append({
            "file": file_path,
            "duration": round(duration, 2),
            "bucket": assigned_bucket,
            "text": example.get("normalized", "")
        })

        counts[assigned_bucket] += 1

        # Stop if we are finished
        if all(c >= CLIPS_PER_BUCKET for c in counts.values()):
            break

    # 8. Save the Reference Sheet
    df = pd.DataFrame(metadata)
    df.to_csv(os.path.join(OUT_DIR, "metadata.csv"), index=False)
    print("\n✅ Dataset Sourced! Check the 'benchmark_data' folder.")

if __name__ == "__main__":
    MY_TOKEN = "YOUR_HUGGINGFACE_TOKEN"
    prepare_folders()
    get_audio_data(MY_TOKEN)

Streaming ai4bharat/IndicVoices (hindi)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Sourcing Audio: 625it [00:05, 109.70it/s]


✅ Dataset Sourced! Check the 'benchmark_data' folder.


# Server


In [ ]:
!VLLM_DISABLE_COMPILE_CACHE=1

In [ ]:
import subprocess
import time
import os
os.environ["VLLM_USE_V1"] = "0"
# Kill existing processes to free up GPU memory
os.system("pkill -f vllm")
time.sleep(2) # Give the GPU a moment to clear

# 1. Added missing backslashes
# 2. Removed duplicate flags
# 3. Added --enforce-eager if CUDAGraphs still fail
launch_cmd = """
vllm serve mistralai/Voxtral-Mini-4B-Realtime-2602 \
    --tokenizer_mode mistral \
    --trust-remote-code \
    --dtype bfloat16 \
    --enable-prefix-caching \
    --gpu-memory-utilization 0.7 \
    --max-model-len 8192 \
    --max-num-batched-tokens 8192 \
    --compilation_config '{"cudagraph_mode": "PIECEWISE"}' \
    --port 8000
"""

with open("vllm_voxtral.log", "w") as f:
    # Use preexec_fn to ensure child processes die if the notebook stops
    subprocess.Popen(launch_cmd, shell=True, stdout=f, stderr=f)

print("🚀 Server is Starting...")
print("📝 Check 'vllm_voxtral.log' for progress. It usually takes 1-2 mins.")

🚀 Server is Starting...
📝 Check 'vllm_voxtral.log' for progress. It usually takes 1-2 mins.


In [ ]:
!tail -20 vllm_voxtral.log

(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /ping, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /invocations, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/chat/completions, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/responses, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/responses/{response_id}, Methods: GET
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/completions, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/messages, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=1169) INFO 03-27 05:30:27 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APISe

# Main Script

In [ ]:
import asyncio
import websockets
import base64
import json
import librosa
import numpy as np
import pandas as pd
import time
import jiwer

# -----------------------------
# CONFIG & DYNAMIC PARAMETERS
# -----------------------------
HOST = "127.0.0.1"
PORT = 8000
MODEL = "mistralai/Voxtral-Mini-4B-Realtime-2602"
CHUNK_SIZE = int(16000 * 2 * STREAMING_DELAY)  # bytes per chunk
METADATA_CSV = "benchmark_data/metadata.csv"

# --- DYNAMIC DELAY ---
# Change this to 0.480 for 480ms, or 0.0 for max speed
STREAMING_DELAY = 2.4

# Text Normalizer for accurate WER
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces()
])

# -----------------------------
# UTILS
# -----------------------------
def audio_to_pcm16_base64(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    pcm16 = (audio * 32767).astype(np.int16)
    return base64.b64encode(pcm16.tobytes()).decode("utf-8")

async def transcribe_clip(audio_path, delay):
    uri = f"ws://{HOST}:{PORT}/v1/realtime"

    # 1. Store absolute timestamps instead of calculating on the fly
    state = {
        "transcription_text": "",
        "start_timestamp": time.time(), # Your start time
        "first_token_timestamp": None,
        "transcription_done_timestamp": None,
        "audio_finished_timestamp": None
    }

    async with websockets.connect(uri) as ws:
        response = json.loads(await ws.recv())
        if response["type"] != "session.created":
            raise RuntimeError(f"Unexpected response: {response}")
        await ws.send(json.dumps({"type": "session.update", "model": MODEL}))

        audio_bytes = base64.b64decode(audio_to_pcm16_base64(audio_path))

        async def send_audio():
            await ws.send(json.dumps({"type": "input_audio_buffer.commit"}))
            for i in range(0, len(audio_bytes), CHUNK_SIZE):
                chunk = audio_bytes[i:i+CHUNK_SIZE]
                await ws.send(json.dumps({
                    "type": "input_audio_buffer.append",
                    "audio": base64.b64encode(chunk).decode("utf-8")
                }))
                await asyncio.sleep(delay)

            await ws.send(json.dumps({"type": "input_audio_buffer.commit", "final": True}))
            # Record exactly when the audio stream ended
            state["audio_finished_timestamp"] = time.time()

        async def receive_results():
            while True:
                resp = json.loads(await ws.recv())

                if resp["type"] == "transcription.delta":
                    # Record the exact moment the first token arrives
                    if state["first_token_timestamp"] is None:
                        state["first_token_timestamp"] = time.time()

                    state["transcription_text"] += resp["delta"]

                elif resp["type"] == "transcription.done":
                    if "text" in resp:
                        state["transcription_text"] = resp["text"]

                    # Record the exact moment it finished
                    state["transcription_done_timestamp"] = time.time()
                    break

                elif resp["type"] == "error":
                    raise RuntimeError(f"Server Error: {resp['error']}")

        # 2. Run both loops concurrently
        await asyncio.gather(send_audio(), receive_results())

        # 3. APPLY YOUR LOGIC SAFELY HERE

        # TTFT: From start of audio to first token
        if state["first_token_timestamp"] is not None:
            ttft = state["first_token_timestamp"] - state["start_timestamp"]
        else:
            ttft = 0.0 # Fallback if no tokens arrived

        # UPL: From end of audio to final transcription
        if state["audio_finished_timestamp"] is not None:
            upl = state["transcription_done_timestamp"] - state["audio_finished_timestamp"]
        else:
            # Fallback in the rare case the model finishes before audio finishes sending
            upl = 0.0

        return state["transcription_text"], ttft, upl

# -----------------------------
# BENCHMARK ENGINE
# -----------------------------
async def run_benchmark():
    df = pd.read_csv(METADATA_CSV)
    final_results = []

    print(f"Starting Benchmark with Delay: {STREAMING_DELAY}s")
    print("-" * 50)

    for bucket in ["5s", "15s", "30s"]:
        subset = df[df.bucket == bucket]
        if subset.empty:
            print(f"No clips for bucket {bucket}, skipping...")
            continue

        ttft_list = []
        upl_list = []
        wer_list = []

        for idx, clip_row in subset.head(8).iterrows():
            clip_path, ref_text = clip_row.file, clip_row.text
            print(f"\n[BUCKET: {bucket}] Processing: {clip_path}")

            clip_ttft = []
            clip_upl = []
            clip_wer = []

            for run_idx in range(3):
                try:
                    raw_text, ttft, upl = await transcribe_clip(clip_path, STREAMING_DELAY)

                    # Convert to ms HERE
                    ttft_ms = ttft * 1000
                    upl_ms = upl * 1000

                    clean_ref = transformation(ref_text)
                    clean_hyp = transformation(raw_text)
                    error_rate = jiwer.wer(clean_ref, clean_hyp)

                    clip_ttft.append(ttft_ms)
                    clip_upl.append(upl_ms)
                    clip_wer.append(error_rate)

                    print(f"  Run {run_idx+1} | TTFT: {ttft_ms:.2f}ms | UPL: {upl_ms:.2f}ms | WER: {error_rate:.3f}")

                except Exception as e:
                    print(f"  Run {run_idx+1} | Failed: {e}")

            # Average per clip
            avg_ttft = np.mean(clip_ttft)
            avg_upl = np.mean(clip_upl)
            avg_wer = np.mean(clip_wer)

            ttft_list.append(avg_ttft)
            upl_list.append(avg_upl)
            wer_list.append(avg_wer)

            print(f"  Clip Average | TTFT: {avg_ttft:.2f}ms | UPL: {avg_upl:.2f}ms | WER: {avg_wer:.3f}")

        # Percentiles (already in ms)
        p50_ttft, p95_ttft = np.percentile(ttft_list, [50, 95])
        p50_upl, p95_upl = np.percentile(upl_list, [50, 95])
        avg_wer_bucket = np.mean(wer_list)

        final_results.append({
            "Bucket": bucket,
            "Delay_ms" : (STREAMING_DELAY*1000),
            "p50_ttft_ms": round(p50_ttft, 2),
            "p95_ttft_ms": round(p95_ttft, 2),
            "p50_upl_ms": round(p50_upl, 2),
            "p95_upl_ms": round(p95_upl, 2),
            "wer": round(avg_wer_bucket, 3)
        })

    # Final summary print (UNCHANGED STYLE)
    summary_df = pd.DataFrame(final_results)
    print("\n" + "="*30 + " FINAL SUMMARY " + "="*30)
    print(summary_df.to_string(index=False))

    # ✅ Save to CSV
    summary_df.to_csv("benchmark_summary.csv", index=False)
    print("\n✅ Results saved to benchmark_summary.csv")

# -----------------------------
# ENTRY POINT
# -----------------------------
if __name__ == "__main__":
    await run_benchmark()

Starting Benchmark with Delay: 2.4s
--------------------------------------------------

[BUCKET: 5s] Processing: benchmark_data/5s/clip_0.wav
  Run 1 | TTFT: 31.82ms | UPL: 273.04ms | WER: 0.227
  Run 2 | TTFT: 26.01ms | UPL: 269.88ms | WER: 0.227
  Run 3 | TTFT: 27.27ms | UPL: 292.20ms | WER: 0.227
  Clip Average | TTFT: 28.36ms | UPL: 278.37ms | WER: 0.227

[BUCKET: 5s] Processing: benchmark_data/5s/clip_1.wav
  Run 1 | TTFT: 30.68ms | UPL: 287.76ms | WER: 0.056
  Run 2 | TTFT: 28.70ms | UPL: 274.75ms | WER: 0.056
  Run 3 | TTFT: 25.29ms | UPL: 269.93ms | WER: 0.056
  Clip Average | TTFT: 28.22ms | UPL: 277.48ms | WER: 0.056

[BUCKET: 5s] Processing: benchmark_data/5s/clip_2.wav
  Run 1 | TTFT: 27.77ms | UPL: 285.51ms | WER: 0.056
  Run 2 | TTFT: 26.44ms | UPL: 265.97ms | WER: 0.056
  Run 3 | TTFT: 25.13ms | UPL: 278.18ms | WER: 0.056
  Clip Average | TTFT: 26.45ms | UPL: 276.55ms | WER: 0.056

[BUCKET: 5s] Processing: benchmark_data/5s/clip_3.wav
  Run 1 | TTFT: 28.18ms | UPL: 292.8